# Stage 2 Evaluation Rollout Analysis

This notebook loads `stage2_eval_rollout.npz`, reconstructs gait phase from `sin_phi` / `cos_phi`, plots the main torque and effort diagnostics, computes the requested summary metrics, and prints a short interpretation.

Note: the rollout does not store physical `dt`, so the time plots use sample index (environment step).

In [ ]:
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

rollout_name = "stage2_eval_rollout.npz"
rollout_path = None  # Optional override, e.g. Path("/absolute/path/to/stage2_eval_rollout.npz")
search_roots = [Path.cwd(), *Path.cwd().parents]

if rollout_path is not None:
    rollout_path = Path(rollout_path).expanduser().resolve()

if rollout_path is None:
    for root in search_roots:
        for candidate in (
            root / rollout_name,
            root / "rl_output" / rollout_name,
            root / "rl" / "rl_output" / rollout_name,
        ):
            if candidate.exists():
                rollout_path = candidate.resolve()
                break
        if rollout_path is not None:
            break

if rollout_path is None:
    for root in search_roots[:3]:
        matches = sorted(root.rglob(rollout_name))
        if matches:
            rollout_path = matches[0].resolve()
            break

if rollout_path is None:
    raise FileNotFoundError(
        "Could not find stage2_eval_rollout.npz. Put it in the repo root, rl_output/, or rl/rl_output/."
    )

rollout_path

In [ ]:
with np.load(rollout_path) as data:
    rollout = {k: np.asarray(data[k]).reshape(-1) for k in data.files}

required = [
    "reward",
    "effort",
    "tau_r",
    "tau_l",
    "hip_r",
    "hip_l",
    "hipd_r",
    "hipd_l",
    "sin_phi",
    "cos_phi",
    "pelvis_vx",
    "torso_pitch",
]
missing = [k for k in required if k not in rollout]
if missing:
    raise KeyError(f"Missing keys: {missing}")

n = len(rollout["reward"])
bad = {k: v.shape for k, v in rollout.items() if len(v) != n}
if bad:
    raise ValueError(f"Array length mismatch: {bad}")

phi = np.mod(np.arctan2(rollout["sin_phi"], rollout["cos_phi"]), 2 * np.pi)
phase_pct = 100 * phi / (2 * np.pi)
sample = np.arange(n)
episode = rollout["episode"] if "episode" in rollout else np.zeros(n, dtype=int)
episode_breaks = np.flatnonzero(np.diff(episode)) + 1

tau_mag = np.sqrt(rollout["tau_r"] ** 2 + rollout["tau_l"] ** 2)

log_path = None
baseline_effort = None
for root in [rollout_path.parent, *rollout_path.parents]:
    for candidate in (
        root / "exo_training_log.json",
        root / "rl_output" / "exo_training_log.json",
        root / "rl" / "rl_output" / "exo_training_log.json",
    ):
        if candidate.exists():
            log_path = candidate.resolve()
            with open(log_path) as f:
                baseline_effort = json.load(f).get("baseline_effort")
            break
    if baseline_effort is not None:
        break

print(f"rollout: {rollout_path}")
print(f"samples: {n}")
print(f"episodes: {len(np.unique(episode))}")
if baseline_effort is not None:
    print(f"baseline effort: {baseline_effort:.6f}  ({log_path})")
else:
    print("baseline effort: not found")

In [ ]:
def phase_bin_mean(phase, values, n_bins=32):
    edges = np.linspace(0, 2 * np.pi, n_bins + 1)
    idx = np.digitize(phase, edges, right=False) - 1
    idx[idx == n_bins] = 0
    counts = np.bincount(idx, minlength=n_bins)
    sums = np.bincount(idx, weights=values, minlength=n_bins)
    means = np.divide(
        sums,
        counts,
        out=np.full(n_bins, np.nan, dtype=float),
        where=counts > 0,
    )
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, means, idx

phase_centers, tau_r_phase_mean, phase_idx = phase_bin_mean(phi, rollout["tau_r"])
_, tau_l_phase_mean, _ = phase_bin_mean(phi, rollout["tau_l"])

pred_tau_r = tau_r_phase_mean[phase_idx]
pred_tau_l = tau_l_phase_mean[phase_idx]

var_tau_r = np.var(rollout["tau_r"])
var_tau_l = np.var(rollout["tau_l"])
phase_lock_r = 1 - np.var(rollout["tau_r"] - pred_tau_r) / var_tau_r if var_tau_r > 0 else np.nan
phase_lock_l = 1 - np.var(rollout["tau_l"] - pred_tau_l) / var_tau_l if var_tau_l > 0 else np.nan

mean_effort = float(np.mean(rollout["effort"]))
std_effort = float(np.std(rollout["effort"]))
mean_torque_magnitude = float(np.mean(tau_mag))
smoothness = float(
    0.5 * (
        np.mean(np.abs(np.diff(rollout["tau_r"])))
        + np.mean(np.abs(np.diff(rollout["tau_l"])))
    )
)
smoothness_ratio = smoothness / max(mean_torque_magnitude, 1e-9)

metrics = {
    "mean effort": mean_effort,
    "std effort": std_effort,
    "mean torque magnitude": mean_torque_magnitude,
    "smoothness (mean |Δτ|)": smoothness,
    "phase-lock score right": float(phase_lock_r),
    "phase-lock score left": float(phase_lock_l),
    "smoothness / mean torque": float(smoothness_ratio),
}

if baseline_effort is not None and baseline_effort > 0:
    metrics["baseline effort"] = float(baseline_effort)
    metrics["effort reduction vs baseline (%)"] = float(
        100 * (baseline_effort - mean_effort) / baseline_effort
    )

for name, value in metrics.items():
    print(f"{name:>32s}: {value: .6f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(sample, rollout["tau_r"], label="tau_r", lw=1.5)
axes[0].plot(sample, rollout["tau_l"], label="tau_l", lw=1.5)
for b in episode_breaks:
    axes[0].axvline(b, color="k", lw=0.6, alpha=0.15)
axes[0].set_ylabel("torque")
axes[0].set_title("Torque vs sample")
axes[0].legend()

axes[1].plot(sample, rollout["effort"], color="tab:green", lw=1.5)
for b in episode_breaks:
    axes[1].axvline(b, color="k", lw=0.6, alpha=0.15)
axes[1].set_xlabel("sample (time step)")
axes[1].set_ylabel("effort")
axes[1].set_title("Effort vs sample")

plt.tight_layout()
plt.show()

In [ ]:
phase_centers_pct = 100 * phase_centers / (2 * np.pi)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)

axes[0].scatter(phase_pct, rollout["tau_r"], s=10, alpha=0.15, color="tab:blue")
axes[0].plot(phase_centers_pct, tau_r_phase_mean, color="black", lw=2)
axes[0].set_title("Right torque vs gait phase")
axes[0].set_xlabel("gait phase (%)")
axes[0].set_ylabel("torque")
axes[0].set_xlim(0, 100)

axes[1].scatter(phase_pct, rollout["tau_l"], s=10, alpha=0.15, color="tab:orange")
axes[1].plot(phase_centers_pct, tau_l_phase_mean, color="black", lw=2)
axes[1].set_title("Left torque vs gait phase")
axes[1].set_xlabel("gait phase (%)")
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bins = 40
axes[0].hist(rollout["tau_r"], bins=bins, alpha=0.65, label="tau_r", color="tab:blue")
axes[0].hist(rollout["tau_l"], bins=bins, alpha=0.65, label="tau_l", color="tab:orange")
axes[0].set_xlabel("torque")
axes[0].set_ylabel("count")
axes[0].set_title("Torque histogram")
axes[0].legend()

axes[1].scatter(rollout["hipd_r"], rollout["tau_r"], s=10, alpha=0.18, label="right", color="tab:blue")
axes[1].scatter(rollout["hipd_l"], rollout["tau_l"], s=10, alpha=0.18, label="left", color="tab:orange")
axes[1].set_xlabel("hip velocity")
axes[1].set_ylabel("torque")
axes[1].set_title("Torque vs hip velocity")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
def strength_label(score):
    if np.isnan(score):
        return "undetermined"
    if score >= 0.60:
        return "strong"
    if score >= 0.30:
        return "moderate"
    return "weak"

phase_lock_mean = np.nanmean([phase_lock_r, phase_lock_l])

if phase_lock_mean >= 0.60:
    periodic_text = "Torque looks clearly periodic across the gait cycle."
elif phase_lock_mean >= 0.30:
    periodic_text = "Torque shows a moderate periodic pattern."
else:
    periodic_text = "Torque looks weakly periodic or only loosely structured."

if min(phase_lock_r, phase_lock_l) >= 0.30:
    alignment_text = "Both legs appear meaningfully aligned to gait phase."
elif max(phase_lock_r, phase_lock_l) >= 0.30:
    alignment_text = "One leg is phase-aligned, but the other is weaker."
else:
    alignment_text = "Phase alignment looks weak."

if baseline_effort is None or baseline_effort <= 0:
    effort_text = "Effort reduction vs baseline cannot be determined from the rollout alone."
else:
    reduction_pct = 100 * (baseline_effort - mean_effort) / baseline_effort
    if reduction_pct > 5:
        effort_text = f"Mean effort is reduced by {reduction_pct:.1f}% versus the saved baseline."
    elif reduction_pct < -5:
        effort_text = f"Mean effort is higher by {-reduction_pct:.1f}% versus the saved baseline."
    else:
        effort_text = f"Mean effort is essentially unchanged ({reduction_pct:.1f}% vs baseline)."

if smoothness_ratio < 0.08:
    smooth_text = "The policy looks very smooth."
elif smoothness_ratio < 0.18:
    smooth_text = "The policy looks reasonably smooth with limited step-to-step jitter."
else:
    smooth_text = "The policy looks noisy relative to its torque magnitude."

print("Interpretation")
print(f"- Is torque periodic? {periodic_text}")
print(f"- Does it align with gait phase? {alignment_text}")
print(f"- Is effort reduced or unchanged? {effort_text}")
print(f"- Is the policy noisy or smooth? {smooth_text}")
print()
print("Phase-lock score = fraction of torque variance explained by gait-phase bins.")
print(f"Phase-lock score (right): {phase_lock_r:.3f} [{strength_label(phase_lock_r)}]")
print(f"Phase-lock score (left):  {phase_lock_l:.3f} [{strength_label(phase_lock_l)}]")
print(f"Smoothness / mean torque: {smoothness_ratio:.3f}")